<p align="center">
  <a href="https://github.com/wavekat/wavekat-lab">
    <img src="https://github.com/wavekat/wavekat-brand/raw/main/assets/banners/wavekat-lab-narrow.svg" alt="WaveKat Lab">
  </a>
</p>

# 06 — Upload two-model-agreed clips to the platform

Take the rows from `04c_models_agree.ipynb` (where our zh checkpoint and
pipecat-v3 agree on the label), rank them **pipecat-first, then ours**,
and post each one to the WaveKat platform as an auto-labelled annotation
with a `confidence` score.

The platform's records UI knows about `source = 'pre_labeled'` and a
per-row `confidence` (see `wavekat-platform/services/api/migrations/0012_annotation_confidence.sql`)
and lets a reviewer triage low-confidence rows first.

## What this notebook needs

1. A platform account with a project the upload should land in, and a
   label set on that project that defines `continuation` and
   `end_of_turn` keys (matching the binary labels we predict here).
2. A `wkcli_…` bearer token. The platform UI does **not** currently
   expose a manual "mint a token" button — tokens are issued through
   the `wk login` loopback flow:
   ```bash
   # Mint a token against your local platform:
   WK_BASE_URL=https://localhost:5020 wk login
   # …or against the deployed one:
   wk login   # defaults to https://platform.wavekat.com
   ```
   `wk login` opens your browser for GitHub auth, the platform mints
   the token, and the CLI persists it to
   `~/Library/Application Support/wavekat/auth.json` (macOS — see
   `wavekat-cli/src/config.rs:30` for the per-OS path).
3. A `.env` (alongside this repo's existing `.env`) with the token
   copied out of `auth.json`. Easiest one-liner:
   ```bash
   WAVEKAT_API_URL=https://localhost:5020
   WAVEKAT_API_TOKEN=$(jq -r .token "$HOME/Library/Application Support/wavekat/auth.json")
   WAVEKAT_PROJECT_ID=<paste from the platform UI>
   ```
   Make sure `WAVEKAT_API_URL` matches the host the token was minted
   against — a token issued by the local platform won't work against
   the production one and vice-versa.
4. The local MagicData-RAMC WAV files. The clip bytes are sliced from
   the same path that `04c` reads.

In [ ]:
from __future__ import annotations

import hashlib
import io
import os
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import soundfile as sf
from tqdm.auto import tqdm

MINING_ROOT  = Path("../../datasets/smart-turn-zh-mining").resolve()
AGREED_IN    = MINING_ROOT / "candidates_models_agree.parquet"
WAV_DIR      = Path("../../datasets/MagicData-RAMC/MDT2021S003/WAV").resolve()

# Load WAVEKAT_* from a `.env` walking up from this notebook (same pattern
# as 05_score_llm.ipynb). `override=False` so an already-exported shell
# value wins over `.env`.
try:
    from dotenv import find_dotenv, load_dotenv
    dotenv_path = find_dotenv(usecwd=True)
    if dotenv_path:
        load_dotenv(dotenv_path, override=False)
        print(f"loaded .env  : {Path(dotenv_path).name} (from {Path(dotenv_path).parent.name}/)")
    else:
        print("loaded .env  : (none found — relying on shell environment)")
except ImportError:
    print("loaded .env  : (python-dotenv not installed — relying on shell environment)")

# Platform endpoint + auth. Default to the local dev URL so a wrong env
# var doesn't accidentally fire writes at production.
PLATFORM_BASE_URL = os.environ.get("WAVEKAT_API_URL", "https://localhost:5020").rstrip("/")
PLATFORM_TOKEN    = os.environ.get("WAVEKAT_API_TOKEN")
PROJECT_ID        = os.environ.get("WAVEKAT_PROJECT_ID")

# Cap the upload run. Default is large enough to cover the full agreed set
# (~couple of thousand rows). Set MAX_UPLOADS in the environment to a
# smaller number when you want a quick smoke test instead of a real run.
MAX_UPLOADS = int(os.environ.get("MAX_UPLOADS", "100000"))

# Map our binary {0, 1} labels to the platform label-set keys. The label
# set on the project must define exactly these keys for the create call to
# succeed.
LABEL_KEY_BY_PRED = {0: "continuation", 1: "end_of_turn"}

print(f"agreed parquet : {AGREED_IN.name} (exists={AGREED_IN.exists()})")
print(f"wav dir        : {WAV_DIR.name}/  (exists={WAV_DIR.exists()})")
print(f"platform       : {PLATFORM_BASE_URL}")
print(f"token set      : {bool(PLATFORM_TOKEN)}")
print(f"project id     : {PROJECT_ID or '(unset — set WAVEKAT_PROJECT_ID)'}")
print(f"max uploads    : {MAX_UPLOADS}")
print("✅ config loaded")

In [2]:
# Ping /api/me — the cheapest possible "does my token work?" check.
# A 401 here means the token is wrong / revoked / minted against a
# different platform host (e.g. token from prod, URL pointing at local).
# Run this on its own first; only continue if it returns your login.
if not PLATFORM_TOKEN:
    raise SystemExit("WAVEKAT_API_TOKEN is unset — mint one with `wk login --base-url …` and export it.")

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {PLATFORM_TOKEN}",
    "User-Agent": "wavekat-lab/06-upload-notebook",
})

me = session.get(f"{PLATFORM_BASE_URL}/api/me", timeout=10)
if me.status_code == 401:
    raise SystemExit(
        f"401 from {PLATFORM_BASE_URL}/api/me — token rejected.\n"
        f"  • Was the token minted against this host? (tokens are scoped to a single platform.)\n"
        f"  • Has it been revoked? Mint a fresh one: `WK_BASE_URL={PLATFORM_BASE_URL} wk login`."
    )
me.raise_for_status()
me_json = me.json()
print(f"authenticated as : {me_json.get('login')}  (role={me_json.get('role')}, id={me_json.get('id')})")
print("✅ token works")

authenticated as : wavekat-eason  (role=root, id=220911746)
✅ token works


In [3]:
# Verify the destination project + its active label set. Split out from the
# /me ping so a project-config issue doesn't masquerade as an auth failure.
if not PROJECT_ID:
    raise SystemExit("WAVEKAT_PROJECT_ID is unset — set it to the destination project's id.")

proj = session.get(f"{PLATFORM_BASE_URL}/api/projects/{PROJECT_ID}", timeout=10)
if proj.status_code == 404:
    raise SystemExit(
        f"404 on project {PROJECT_ID} — either the id is wrong, or your account "
        f"isn't a member of that project (the API returns 404, not 403, to avoid leaking existence)."
    )
proj.raise_for_status()
proj_json = proj.json()
print(f"project          : {proj_json.get('name')}  (role={proj_json.get('myRoleInProject')})")
active_label_set_id = proj_json.get("activeLabelSetId")
if not active_label_set_id:
    raise SystemExit("project has no active label set — pick one in the platform UI first.")

ls = session.get(f"{PLATFORM_BASE_URL}/api/label-sets/{active_label_set_id}", timeout=10)
ls.raise_for_status()
ls_json = ls.json()
label_keys = {l["key"] for l in ls_json.get("labels", [])}
missing = set(LABEL_KEY_BY_PRED.values()) - label_keys
if missing:
    raise SystemExit(
        f"active label set '{ls_json.get('name')}' is missing required keys: {sorted(missing)}"
    )
print(f"label set        : {ls_json.get('name')}  (keys: {sorted(label_keys)})")
print("✅ project + label set verified")

project          : Local Two  (role=root)
label set        : Turn Detection V1  (keys: ['continuation', 'end_of_turn'])
✅ project + label set verified


In [4]:
# Load 04c's output and re-derive the pipecat-first ranking. We rank the
# *agreed* set in two halves so EOT and CONT are picked top-down by their
# respective model probability instead of mixing the two scales.
agreed = pd.read_parquet(AGREED_IN)
print(f"agreed rows: {len(agreed)}  (cols: {list(agreed.columns)})")

agreed_eot  = agreed[agreed["model_pred"] == 1].sort_values(
    ["pipecat_prob", "model_prob"], ascending=[False, False]
).reset_index(drop=True)
agreed_cont = agreed[agreed["model_pred"] == 0].sort_values(
    ["pipecat_prob", "model_prob"], ascending=[True, True]
).reset_index(drop=True)

# Confidence = joint probability of the *chosen* label under both models.
# For EOT rows: P(EOT | pipecat) * P(EOT | ours) = pipecat_prob * model_prob.
# For CONT rows: P(CONT | pipecat) * P(CONT | ours) = (1 - pipecat_prob) * (1 - model_prob).
# Treating the two models as independent yes/no votes; the product collapses
# to a single [0, 1] number that the records-UI confidence column can rank by.
# Both factors are already > 0.5 inside the agreed set (otherwise the models
# wouldn't agree on this label), so the product stays well above chance.
pp_eot = agreed_eot["pipecat_prob"].astype(float)
mp_eot = agreed_eot["model_prob"].astype(float)
agreed_eot["confidence"] = pp_eot * mp_eot

pp_cont = agreed_cont["pipecat_prob"].astype(float)
mp_cont = agreed_cont["model_prob"].astype(float)
agreed_cont["confidence"] = (1.0 - pp_cont) * (1.0 - mp_cont)

# Interleave EOT-best with CONT-best so a small MAX_UPLOADS still ends up
# with a balanced sample. Reset the index after concat for the head() cap.
max_per_side = (MAX_UPLOADS + 1) // 2
to_upload = pd.concat([
    agreed_eot.head(max_per_side),
    agreed_cont.head(max_per_side),
], ignore_index=True).head(MAX_UPLOADS)
print(f"queued for upload: {len(to_upload)}  "
      f"(EOT={int((to_upload['model_pred']==1).sum())}, "
      f"CONT={int((to_upload['model_pred']==0).sum())})")
print(f"confidence range : "
      f"min={to_upload['confidence'].min():.3f}, "
      f"median={to_upload['confidence'].median():.3f}, "
      f"max={to_upload['confidence'].max():.3f}")
to_upload.head()

agreed rows: 21811  (cols: ['session_id', 'candidate_idx', 'parent_utt_idx', 'source', 'label', 'clip_start_s', 'clip_end_s', 'clip_duration_s', 'speaker_id', 'gender', 'text', 'parent_text', 'gap_to_next_s', 'next_speaker_id', 'vad_trim_offset_ms', 'model_prob', 'model_pred', 'pipecat_prob', 'pipecat_pred'])
queued for upload: 21811  (EOT=15367, CONT=6444)
confidence range : min=0.255, median=0.554, max=0.643


,session_id,candidate_idx,parent_utt_idx,source,label,clip_start_s,clip_end_s,clip_duration_s,speaker_id,gender,text,parent_text,gap_to_next_s,next_speaker_id,vad_trim_offset_ms,model_prob,model_pred,pipecat_prob,pipecat_pred,confidence
0,CTS-CN-F2F-2019-11-15-528,36,211,speaker_change,1,478.856,482.912,4.056,G00000274,女,我就感觉啊，好多事情好像都是年龄越大，胆子越小,我就感觉啊，好多事情好像都是年龄越大，胆子越小,0.708,G00000275,-4.0,0.612894,1,0.99013,1,0.606844
1,CTS-CN-F2F-2019-11-15-1477,49,179,speaker_change,1,564.986,567.360,2.374,G00000312,男,毒品是法律中严禁的,毒品是法律中严禁的,0.334,G00000313,-2.0,0.609752,1,0.99013,1,0.603733
2,CTS-CN-F2F-2019-11-15-525,86,568,intra_utterance_cut,0,1465.008,1467.648,2.640,G00000621,男,那像打王者的话，他,那像打王者的话，他们他们也打,NaN,,-172.7,0.596036,1,0.99013,1,0.590153
3,CTS-CN-F2F-2019-11-15-649,0,4,intra_utterance_cut,0,12.577,14.912,2.335,G00000088,男,呃，丹顶鹤我想大家应该都知,呃，丹顶鹤我想大家应该都知道，它是国家一级保护动物,NaN,,78.0,0.581866,1,0.99013,1,0.576122
4,CTS-CN-F2F-2019-11-15-88,76,262,speaker_change,1,862.176,864.416,2.240,G00000155,男,三步一个。口鼻呼吸一次。,三步一个。口鼻呼吸一次。,0.448,G00000154,0.0,0.561495,1,0.99013,1,0.555953


In [5]:
# Per-session helpers: hash the WAV once (the platform dedupes by
# sha256), and remember the file row id so we don't refetch on every
# clip. The `files` POST is idempotent so a re-run is cheap, but a single
# session yields 50+ clips — caching keeps the run snappy.
import functools

_HASH_CHUNK = 1024 * 1024  # 1 MB — keep memory bounded for large session WAVs.

@functools.lru_cache(maxsize=None)
def session_wav_meta(session_id: str) -> dict:
    path = WAV_DIR / f"{session_id}.wav"
    info = sf.info(str(path))
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(_HASH_CHUNK)
            if not chunk:
                break
            h.update(chunk)
    return {
        "path": path,
        "sha256": h.hexdigest(),
        "sample_rate": int(info.samplerate),
        "channels": int(info.channels),
        "duration_sec": float(info.frames) / float(info.samplerate),
    }

@functools.lru_cache(maxsize=None)
def upsert_file(session_id: str) -> str:
    """Create-or-get the platform `files` row for a session WAV; return its id."""
    meta = session_wav_meta(session_id)
    body = {
        "name": session_id,
        "originalFilename": f"{session_id}.wav",
        "sha256": meta["sha256"],
        "durationSec": meta["duration_sec"],
        "sampleRate": meta["sample_rate"],
        "channelCount": meta["channels"],
    }
    r = session.post(
        f"{PLATFORM_BASE_URL}/api/projects/{PROJECT_ID}/files",
        json=body, timeout=15,
    )
    r.raise_for_status()
    j = r.json()
    return j["id"]

def slice_wav_bytes(session_id: str, start_s: float, end_s: float) -> bytes:
    """Read [start_s, end_s) from the source WAV and re-emit a 16-bit PCM
    WAV blob. Mono-mixes the source so the clip stays small and the
    server-side WAV-meta parse stays in the supported codec list."""
    meta = session_wav_meta(session_id)
    sr = meta["sample_rate"]
    start  = max(0, int(start_s * sr))
    frames = max(1, int((end_s - start_s) * sr))
    audio, _ = sf.read(str(meta["path"]), start=start, frames=frames,
                       dtype="float32", always_2d=False)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    buf = io.BytesIO()
    sf.write(buf, audio, sr, format="WAV", subtype="PCM_16")
    return buf.getvalue()

print("✅ helpers ready")

✅ helpers ready


In [ ]:
# Upload loop. Each row: ensure the session file exists on the platform,
# create the annotation row with source=pre_labeled + confidence, then
# PUT the sliced WAV bytes. Every step talks to the same Worker — failures
# are surfaced row-by-row so a one-off 4xx doesn't tank the whole run.
#
# Range is `[clip_start_s, clip_end_s]` straight from the parquet — the
# same slice that 04c plays in its audio embeds and that the model scored
# on. `clip_start_s` is already snapped to the parent utterance's start
# (or `clip_end_s - 8s`, whichever is later) by 03_build_candidates, so
# no widening here.
#
# Progress: tqdm bar shows rows/s and ETA; the postfix carries running
# ok / fail counts so a stalled bar (e.g. all uploads failing) is obvious
# without scrolling to the results dataframe at the bottom.
results: list[dict] = []
ok_count = 0
fail_count = 0

pbar = tqdm(to_upload.iterrows(), total=len(to_upload), desc="upload", unit="clip")
for i, row in pbar:
    sid = str(row["session_id"])
    pred = int(row["model_pred"])
    label_key = LABEL_KEY_BY_PRED[pred]
    confidence = float(row["confidence"])
    start_s = float(row["clip_start_s"])
    end_s   = float(row["clip_end_s"])
    asr_text = str(row.get("text", "") or "")[:4000]

    try:
        file_id = upsert_file(sid)
        create_body = {
            "labelKey":   label_key,
            "startSec":   start_s,
            "endSec":     end_s,
            "asrText":    asr_text or None,
            "source":     "pre_labeled",
            "confidence": confidence,
            "notes": (
                f"smart-turn-mining 04c agreement\n"
                f"ours_prob={float(row['model_prob']):.4f}, "
                f"pipecat_prob={float(row['pipecat_prob']):.4f}, "
                f"gold_label={row.get('label')}"
            ),
        }
        r = session.post(
            f"{PLATFORM_BASE_URL}/api/files/{file_id}/annotations",
            json=create_body, timeout=15,
        )
        r.raise_for_status()
        ann = r.json()
        ann_id = ann["id"]

        wav_bytes = slice_wav_bytes(sid, start_s, end_s)
        u = session.put(
            f"{PLATFORM_BASE_URL}/api/annotations/{ann_id}/clip",
            data=wav_bytes,
            headers={"Content-Type": "audio/wav"},
            timeout=30,
        )
        u.raise_for_status()
        results.append({
            "ok": True, "annotation_id": ann_id, "label": label_key,
            "confidence": confidence, "bytes": len(wav_bytes),
            "session": sid, "range": f"{start_s:.2f}-{end_s:.2f}",
        })
        ok_count += 1
    except requests.HTTPError as e:
        body = getattr(e.response, "text", "")
        results.append({
            "ok": False, "label": label_key, "confidence": confidence,
            "session": sid, "range": f"{start_s:.2f}-{end_s:.2f}",
            "error": f"HTTP {e.response.status_code}: {body[:200]}",
        })
        fail_count += 1
    except Exception as e:
        results.append({
            "ok": False, "label": label_key, "confidence": confidence,
            "session": sid, "range": f"{start_s:.2f}-{end_s:.2f}",
            "error": f"{type(e).__name__}: {e}",
        })
        fail_count += 1
    pbar.set_postfix(ok=ok_count, fail=fail_count, refresh=False)

results_df = pd.DataFrame(results)
print(f"uploaded ok : {ok_count}")
print(f"failed      : {fail_count}")
if fail_count:
    # Surface the first few failures inline — much easier than hunting
    # through a 1000-row results_df for the broken ones.
    print("\nfirst failures:")
    display(results_df[~results_df["ok"]].head(5))
results_df.head(20)

In [ ]:
# Sanity-check: re-fetch the project's auto-labelled rows ordered by
# confidence ASC (the records UI's triage view) and show the lowest-
# confidence ones first. If the upload landed, these should include rows
# we just wrote.
r = session.get(
    f"{PLATFORM_BASE_URL}/api/projects/{PROJECT_ID}/annotations",
    params={
        "page": 1,
        "pageSize": 20,
        "source": "auto",
        "orderBy": "confidence",
        "order": "asc",
    },
    timeout=15,
)
r.raise_for_status()
page = r.json()
print(f"project auto-labelled total : {page['total']}")
if page['annotations']:
    cols = ["labelKey", "confidence", "reviewStatus", "startSec", "endSec", "fileName"]
    display(pd.DataFrame(page['annotations'])[cols].head(20))
print("✅ verified via the records-list endpoint")